# DATA PREPARATION AND CLEANING

## Load the datasets

In [17]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path("../data_sets")
 
train_features_df = pd.read_csv(
    DATA_DIR / "training_set_features.csv"
)

train_labels_df = pd.read_csv(
    DATA_DIR / "training_set_labels.csv"
)

test_features_df = pd.read_csv(
    DATA_DIR / "test_set_features.csv"
)
 

print("Training features:", train_features_df.shape)
print("Training labels:", train_labels_df.shape)
print("Test features:", test_features_df.shape)
 

Training features: (26707, 36)
Training labels: (26707, 3)
Test features: (26708, 36)


## DATA SET INSPECTION -- understanding the dataset.

### Step 1: Viewing the names of the columns in the datasets

In [18]:
print("Training feature columns:")
print(train_features_df.columns.tolist())

print("\nTraining label columns:")
print(train_labels_df.columns.tolist())

Training feature columns:
['respondent_id', 'h1n1_concern', 'h1n1_knowledge', 'behavioral_antiviral_meds', 'behavioral_avoidance', 'behavioral_face_mask', 'behavioral_wash_hands', 'behavioral_large_gatherings', 'behavioral_outside_home', 'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal', 'chronic_med_condition', 'child_under_6_months', 'health_worker', 'health_insurance', 'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk', 'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective', 'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'age_group', 'education', 'race', 'sex', 'income_poverty', 'marital_status', 'rent_or_own', 'employment_status', 'hhs_geo_region', 'census_msa', 'household_adults', 'household_children', 'employment_industry', 'employment_occupation']

Training label columns:
['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']


### Step 2 Check data types and the missing values

In [19]:
## Checking the structure of the train_feature dataset
train_features_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   respondent_id                26707 non-null  int64  
 1   h1n1_concern                 26615 non-null  float64
 2   h1n1_knowledge               26591 non-null  float64
 3   behavioral_antiviral_meds    26636 non-null  float64
 4   behavioral_avoidance         26499 non-null  float64
 5   behavioral_face_mask         26688 non-null  float64
 6   behavioral_wash_hands        26665 non-null  float64
 7   behavioral_large_gatherings  26620 non-null  float64
 8   behavioral_outside_home      26625 non-null  float64
 9   behavioral_touch_face        26579 non-null  float64
 10  doctor_recc_h1n1             24547 non-null  float64
 11  doctor_recc_seasonal         24547 non-null  float64
 12  chronic_med_condition        25736 non-null  float64
 13  child_under_6_months       

### Learning from .info()

- The dataset has 26,707 rows and 36 columns. (that means 1 ID and 35 Predictor columns)
- respondent_id is complete.
- Several predictor columns contain missing values.
- Occupation, industry, and health insurance have especially high missingness.
- The dataset contains numeric and categorical variables.
- Many float64 columns may actually represent binary or ordinal categories.

In [ ]:
## Checking the structure of the train_label dataset
train_labels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   respondent_id     26707 non-null  int64
 1   h1n1_vaccine      26707 non-null  int64
 2   seasonal_vaccine  26707 non-null  int64
dtypes: int64(3)
memory usage: 626.1 KB


In [24]:

# Statistical and categorical summary
display(train_features_df.describe(include="all").T)




,count,unique,top,freq,mean,std,min,25%,50%,75%,max
respondent_id,26707.0,NaN,NaN,NaN,13353.0,7709.791156,0.0,6676.5,13353.0,20029.5,26706.0
h1n1_concern,26615.0,NaN,NaN,NaN,1.618486,0.910311,0.0,1.0,2.0,2.0,3.0
h1n1_knowledge,26591.0,NaN,NaN,NaN,1.262532,0.618149,0.0,1.0,1.0,2.0,2.0
behavioral_antiviral_meds,26636.0,NaN,NaN,NaN,0.048844,0.215545,0.0,0.0,0.0,0.0,1.0
behavioral_avoidance,26499.0,NaN,NaN,NaN,0.725612,0.446214,0.0,0.0,1.0,1.0,1.0
behavioral_face_mask,26688.0,NaN,NaN,NaN,0.068982,0.253429,0.0,0.0,0.0,0.0,1.0
behavioral_wash_hands,26665.0,NaN,NaN,NaN,0.825614,0.379448,0.0,1.0,1.0,1.0,1.0
behavioral_large_gatherings,26620.0,NaN,NaN,NaN,0.35864,0.47961,0.0,0.0,0.0,1.0,1.0
behavioral_outside_home,26625.0,NaN,NaN,NaN,0.337315,0.472802,0.0,0.0,0.0,1.0,1.0
behavioral_touch_face,26579.0,NaN,NaN,NaN,0.677264,0.467531,0.0,0.0,1.0,1.0,1.0


In [28]:
print(
    "Missing IDs in features:",
    train_features_df["respondent_id"].isna().sum()
)

print(
    "Missing IDs in labels:",
    train_labels_df["respondent_id"].isna().sum()
)

Missing IDs in features: 0
Missing IDs in labels: 0


### Checking Duplicate IDs

In [29]:
print(
    "Duplicate IDs in features:",
    train_features_df["respondent_id"].duplicated().sum()
)

print(
    "Duplicate IDs in labels:",
    train_labels_df["respondent_id"].duplicated().sum()
)

Duplicate IDs in features: 0
Duplicate IDs in labels: 0


### Check completely duplicated rows

In [30]:
print(
    "Duplicate feature rows:",
    train_features_df.duplicated().sum()
)

print(
    "Duplicate label rows:",
    train_labels_df.duplicated().sum()
)

Duplicate feature rows: 0
Duplicate label rows: 0


### Check whether the IDs match across files

In [31]:
feature_ids = set(train_features_df["respondent_id"])
label_ids = set(train_labels_df["respondent_id"])

print(
    "Feature IDs without labels:",
    len(feature_ids - label_ids)
)

print(
    "Label IDs without features:",
    len(label_ids - feature_ids)
)

Feature IDs without labels: 0
Label IDs without features: 0


### Learning from checks

- no respondent ID is missing;
- no respondent appears twice;
- no complete row is duplicated;
- every feature row has a matching label row.

### Validating Target columns

In [38]:
target_columns = [
    "h1n1_vaccine",
    "seasonal_vaccine"
]

print("Missing target values:")
print(train_labels_df[target_columns].isna().sum())

Missing target values:
h1n1_vaccine        0
seasonal_vaccine    0
dtype: int64


In [37]:
## contain only 0 and 1.
valid_targets = (
    train_labels_df[target_columns]
    .isin([0, 1])
    .all()
    .all()
)

print("All target values are valid:", valid_targets)

All target values are valid: True


## MERGING TRAINING_SET_FEATURES AND TRAINING_SET_LABELS

In [45]:
merged_train_data_df = train_features_df.merge(
    train_labels_df,
    on="respondent_id",
    how="inner",
    validate="one_to_one"
)

merged_train_data_df


,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,NaN,NaN,0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,NaN,NaN,0,1
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26702,26702,2.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,Own,Not in Labor Force,qufhixun,Non-MSA,0.0,0.0,NaN,NaN,0,0
26703,26703,1.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,Rent,Employed,lzgpxyit,"MSA, Principle City",1.0,0.0,fcxhlnwr,cmhcxjea,0,0
26704,26704,2.0,2.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,...,Own,NaN,lzgpxyit,"MSA, Not Principle City",0.0,0.0,NaN,NaN,0,1
26705,26705,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,...,Rent,Employed,lrircsnp,Non-MSA,1.0,0.0,fcxhlnwr,haliazsg,0,0


### Checking if the merge was correctly done

In [ ]:
print("Features shape:", train_features_df.shape)
print("Labels shape:", train_labels_df.shape)
print("Merged shape:", merged_train_data_df.shape)

Features shape: (26707, 36)
Labels shape: (26707, 3)
Merged shape: (26707, 38)


### Checking for Missing Values

In [60]:
# Count missing values
missing_count = merged_train_data_df.isna().sum()

# Create summary for columns that contain missing values
missing_summary = pd.DataFrame({
    "missing_count": missing_count[missing_count > 0]
})

# Calculate missing percentages
missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(merged_train_data_df)
    * 100
).round(2)

# Classify the percentages
missing_summary["missingness_level"] = pd.cut(
    missing_summary["missing_percentage"],
    bins=[0, 5, 20, 50, 100],
    labels=["Low", "Moderate", "High", "Very high"],
    right=False
)

# Sort and display
missing_summary = missing_summary.sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)

,missing_count,missing_percentage,missingness_level
employment_occupation,13470,50.44,Very high
employment_industry,13330,49.91,High
health_insurance,12274,45.96,High
income_poverty,4423,16.56,Moderate
doctor_recc_h1n1,2160,8.09,Moderate
doctor_recc_seasonal,2160,8.09,Moderate
rent_or_own,2042,7.65,Moderate
employment_status,1463,5.48,Moderate
marital_status,1408,5.27,Moderate
education,1407,5.27,Moderate


### Findings from Missing Value Inspection

- Missing values occur mainly in employment occupation, employment industry, and health insurance. 
- No target values are missing.
- To avoid losing a large number of respondents, rows will be retained.
- Missing categorical values will receive an explicit unknown category,
- while numeric imputation will be handled during modeling.

## IDENTIFYING categorical and numeric columns

In [68]:
##  EXCCLUDED BECAUSE THEY ARE COMPLETE

excluded_columns = [
    "respondent_id",
    "h1n1_vaccine",
    "seasonal_vaccine"
]

##  ONLY CATEGORICAL COLUMNS
categorical_columns = (
    merged_train_data_df
    .select_dtypes(include=["object", "string"])
    .columns
)

##  ONLY NUMERIC COLUMNS
numeric_columns = (
    merged_train_data_df
    .select_dtypes(include="number")
    .columns
    .difference(excluded_columns)
)

### checking for spelling and capitalisaiton differences

In [70]:
issues_found = False

for column in categorical_columns:
    values = merged_train_data_df[column].dropna()

    # Check leading or trailing spaces
    space_issues = values[
        values != values.str.strip()
    ]

    # Normalize values for comparison
    comparison = pd.DataFrame({
        "original": values,
        "normalized": values.str.strip().str.lower()
    }).drop_duplicates()

    # Find multiple original versions of the same normalized value
    variants = (
        comparison.groupby("normalized")["original"]
        .nunique()
    )

    variants = variants[variants > 1]

    if not space_issues.empty:
        print(
            f"{column}: {len(space_issues)} values "
            "contain surrounding spaces"
        )
        issues_found = True

    if not variants.empty:
        print(
            f"{column}: capitalization or spelling "
            "variants found"
        )

        display(
            comparison[
                comparison["normalized"].isin(
                    variants.index
                )
            ].sort_values("normalized")
        )

        issues_found = True

if not issues_found:
    print(
        "No capitalization or surrounding-space "
        "problems found."
    )

No capitalization or surrounding-space problems found.


In [80]:
numeric_columns = merged_train_data_df.select_dtypes(
    include="number"
).columns

for column in numeric_columns:
    if column != "respondent_id":
        values = sorted(
            merged_train_data_df[column]
            .dropna()
            .unique()
        )

        print(f"{column}: {values}")

h1n1_concern: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]
h1n1_knowledge: [np.float64(0.0), np.float64(1.0), np.float64(2.0)]
behavioral_antiviral_meds: [np.float64(0.0), np.float64(1.0)]
behavioral_avoidance: [np.float64(0.0), np.float64(1.0)]
behavioral_face_mask: [np.float64(0.0), np.float64(1.0)]
behavioral_wash_hands: [np.float64(0.0), np.float64(1.0)]
behavioral_large_gatherings: [np.float64(0.0), np.float64(1.0)]
behavioral_outside_home: [np.float64(0.0), np.float64(1.0)]
behavioral_touch_face: [np.float64(0.0), np.float64(1.0)]
doctor_recc_h1n1: [np.float64(0.0), np.float64(1.0)]
doctor_recc_seasonal: [np.float64(0.0), np.float64(1.0)]
chronic_med_condition: [np.float64(0.0), np.float64(1.0)]
child_under_6_months: [np.float64(0.0), np.float64(1.0)]
health_worker: [np.float64(0.0), np.float64(1.0)]
health_insurance: [np.float64(0.0), np.float64(1.0)]
opinion_h1n1_vacc_effective: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.floa

### CREATING A CLEAN COPY

In [74]:
cleaned_train_data_df = merged_train_data_df.copy()
cleaned_train_data_df.shape

(26707, 38)

### Handle categorical missing values

In [75]:
cleaned_train_data_df[categorical_columns] = (
    cleaned_train_data_df[categorical_columns]
    .fillna("Unknown")
)

### Handle numeric missing values

In [76]:
for column in numeric_columns:
    median_value = cleaned_train_data_df[column].median()

    cleaned_train_data_df[column] = (
        cleaned_train_data_df[column]
        .fillna(median_value)
    )

In [79]:
print(
    "Total missing values:",
    cleaned_train_data_df.isna().sum().sum()
)


print("Before cleaning:", merged_train_data_df.shape)
print("After cleaning:", cleaned_train_data_df.shape)
cleaned_train_data_df.head()

Total missing values: 0
Before cleaning: (26707, 38)
After cleaning: (26707, 38)


,respondent_id,h1n1_concern,h1n1_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,rent_or_own,employment_status,hhs_geo_region,census_msa,household_adults,household_children,employment_industry,employment_occupation,h1n1_vaccine,seasonal_vaccine
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,Own,Not in Labor Force,oxchjgsf,Non-MSA,0.0,0.0,Unknown,Unknown,0,0
1,1,3.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Rent,Employed,bhuqouqj,"MSA, Not Principle City",0.0,0.0,pxcmvdjn,xgwztkwe,0,1
2,2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",2.0,0.0,rucpziij,xtkaffoo,0,0
3,3,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,Rent,Not in Labor Force,lrircsnp,"MSA, Principle City",0.0,0.0,Unknown,Unknown,0,1
4,4,2.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,Own,Employed,qufhixun,"MSA, Not Principle City",1.0,0.0,wxleyezf,emcorrxb,0,0
